# 01 — Ingestão e embeddings locais

Este notebook transforma o corpus estático do RAGnaldo em um índice vetorial persistente. O objetivo é executar a etapa pesada uma vez, fora do startup da aplicação.

## Fluxo

1. descobrir PDFs e documentos autorais;
2. extrair texto preservando fonte e página;
3. produzir chunks com sobreposição;
4. gerar embeddings locais pelo wrapper do LangChain;
5. salvar FAISS, docstore e manifesto com hashes.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
SRC = PROJECT_ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from ragnaldo.config import (
    AUTHORIAL_SOURCES_DIR,
    PROCESSED_DATA_DIR,
    RAW_DATA_DIR,
    SETTINGS,
    VECTOR_STORE_DIR,
)
from ragnaldo.ingestion import (
    build_index,
    discover_sources,
    load_sources,
    split_documents,
)

SETTINGS

/home/f_szekut/projects/tech_builder/ragnaldo/env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


RagSettings(embedding_model='sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2', chunk_size=1000, chunk_overlap=150, retrieval_k=4, device='cpu')

## 1. Inventário das fontes

Arquivos oficiais baixados ficam em `data/raw` e não são versionados. Os documentos autorais ficam em `docs/sources`. O HTML bruto é mantido como evidência da pesquisa, mas a primeira versão do índice usa PDF, Markdown e TXT.

In [2]:
source_paths = discover_sources(RAW_DATA_DIR, PROCESSED_DATA_DIR, AUTHORIAL_SOURCES_DIR)

for path in source_paths:
    print(f'{path.suffix.lower():>5}  {path.relative_to(PROJECT_ROOT)}')

print(f'\n{len(source_paths)} fontes prontas para ingestão')

  .md  data/processed/challenge_trello.md
  .md  data/processed/guia_imersao_one.md
 .pdf  data/raw/manual_one_historico.pdf
.html  data/raw/one_ai_for_tech_oracle_snapshot.html
 .pdf  data/raw/oracle_ai_insights_america_latina_2025.pdf
  .md  docs/sources/challenge_alura_agente.md
  .md  docs/sources/dossie_ragnaldo.md
  .md  docs/sources/fontes_e_licencas.md
  .md  docs/sources/guia_one_ai_for_tech.md

9 fontes prontas para ingestão


### Atenção ao manual histórico

O arquivo `manual_one_historico.pdf` está disponível para pesquisa histórica, mas não deve fundamentar regras atuais. Para o primeiro índice, vamos removê-lo explicitamente do conjunto.

In [3]:
active_sources = [
    path for path in source_paths
    if path.name != 'manual_one_historico.pdf'
]

for path in active_sources:
    print(path.relative_to(PROJECT_ROOT))

data/processed/challenge_trello.md
data/processed/guia_imersao_one.md
data/raw/one_ai_for_tech_oracle_snapshot.html
data/raw/oracle_ai_insights_america_latina_2025.pdf
docs/sources/challenge_alura_agente.md
docs/sources/dossie_ragnaldo.md
docs/sources/fontes_e_licencas.md
docs/sources/guia_one_ai_for_tech.md


## 2. Carregamento

`PyPDFLoader` cria um documento por página. Os arquivos Markdown são carregados como documentos textuais. Cada documento recebe nome, caminho e SHA-256 da fonte.

In [4]:
documents = load_sources(active_sources)
print(f'{len(documents)} documentos/paginas carregados')

{k: v for k, v in documents[0].metadata.items() if k != 'source_path'}

36 documentos/paginas carregados


{'source': 'challenge_trello.md',
 'location': 'documento',
 'source_sha256': '1a1d41493b7a1cac38b823a10f2bd4c0bea28d1dbafd9dfabd2da96daee791bb',
 'format': 'md'}

## 3. Chunking

A configuração inicial usa 1.000 caracteres com 150 de overlap. Esses valores são hipótese, não verdade universal: serão avaliados no notebook 04. Cada chunk recebe um identificador determinístico.

In [5]:
chunks = split_documents(documents)
print(f'{len(chunks)} chunks produzidos')

# Exemplo tirado de fonte autoral: conteudo de terceiros fica fora do
# repositorio publico, inclusive nos outputs versionados do notebook.
exemplo = next(c for c in chunks if c.metadata['source'] == 'dossie_ragnaldo.md')
print({k: v for k, v in exemplo.metadata.items() if k != 'source_path'})
print(exemplo.page_content[:400])

180 chunks produzidos
{'source': 'dossie_ragnaldo.md', 'location': 'documento', 'source_sha256': '1cd5dad17e60ede17fb1e8e9549b70f985d1bf5d56fe443602fce84d1d065856', 'format': 'md', 'start_index': 0, 'chunk_id': '09b95f12a1f3fa8f', 'chunk_position': 166}
# Dossiê técnico do RAGnaldo

## Resumo

RAGnaldo é um guia independente, útil e bem-humorado sobre o ONE AI for Tech, a jornada Tech Builder e a engenharia do próprio agente. O nome combina RAG com Ronaldo. O trocadilho foi criado durante o planejamento do projeto.

Sua personalidade pode fazer comentários leves, mas suas respostas factuais devem estar fundamentadas nos documentos recuperados. Qu


## 4. Embeddings e índice

O wrapper `HuggingFaceEmbeddings` do LangChain carrega o modelo local em CPU. FAISS usa produto interno sobre vetores normalizados, equivalente à similaridade de cosseno para comparação de ranking.

In [6]:
manifest = build_index(chunks, destination=VECTOR_STORE_DIR)
manifest

Loading weights:   0%|                                                       | 0/199 [00:00<?, ?it/s]

Loading weights:  67%|████████████████████████████▉              | 134/199 [00:00<00:00, 1310.88it/s]

Loading weights: 100%|███████████████████████████████████████████| 199/199 [00:00<00:00, 1523.75it/s]

{'created_at': '2026-08-02T21:14:44.794179+00:00',
 'embedding_model': 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2',
 'chunk_size': 1000,
 'chunk_overlap': 150,
 'chunk_count': 180,
 'source_hashes': {'challenge_trello.md': '1a1d41493b7a1cac38b823a10f2bd4c0bea28d1dbafd9dfabd2da96daee791bb',
  'guia_imersao_one.md': '0d57d7aa9c7cc99d254d7824f93825c5a910e304923ae15d749af7827f7705ce',
  'one_ai_for_tech_oracle_snapshot.html': '1133e25062df5ab06542cf292b929ed500ff55c85306339c0531e80e65228be3',
  'oracle_ai_insights_america_latina_2025.pdf': '3bb44d33d3f0354f44fe5843d502f16e2ba5cad6fc3a6e9bfe4558ec1a0e3505',
  'challenge_alura_agente.md': '9c9eb30fabb637d074f09373d14a4147317374305772c1593d8315b143ea1516',
  'dossie_ragnaldo.md': '1cd5dad17e60ede17fb1e8e9549b70f985d1bf5d56fe443602fce84d1d065856',
  'fontes_e_licencas.md': 'a52d3889e6d4062ee0c633a90f38efc6a07f3ec89760d71931edec35352628e7',
  'guia_one_ai_for_tech.md': '616b3e2be25c5d7c9d63d1cba68c3533304f8788e91704ae6a3f8709f

## Resultado esperado

A pasta `artifacts/vector_store` passa a conter `index.faiss`, `index.pkl` e `artifact_manifest.json`. Os arquivos permanecem fora do Git até definirmos a estratégia de build/deploy. O manifesto permite detectar alteração antes da desserialização do docstore.